In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path

drive_folder = Path("/content/drive/MyDrive")

excel_files = [
    f for f in drive_folder.rglob("*")
    if f.is_file() and f.suffix.lower() in [".xlsx", ".xls"]
]

print("Excel files found in My Drive:")

for f in excel_files:
    print(f)

Excel files found in My Drive:


In [5]:
from pathlib import Path

project_folder = Path(
    "/content/drive/MyDrive/Statistical Modelling Project"
)

print("Project folder exists:", project_folder.exists())

# Find all Excel files inside your project folder
excel_files = []

for file in project_folder.rglob("*"):
    if file.is_file() and file.suffix.lower() in [".xlsx", ".xls"]:
        excel_files.append(file)

print("\nExcel files found:")

for file in excel_files:
    print(file)

Project folder exists: True

Excel files found:


In [6]:

from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount('/content/drive')

# Find the Excel workbook in Google Drive
matches = list(
    Path('/content/drive/MyDrive').rglob('online_retail_II.xlsx')
)

if not matches:
    raise FileNotFoundError(
        "online_retail_II.xlsx was not found in My Drive. "
        "Check its location and filename."
    )

file_name = str(matches[0])

print("Excel file found:", file_name)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: online_retail_II.xlsx was not found in My Drive. Check its location and filename.

In [7]:
file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

df_2009_2010 = pd.read_excel(
    file_name,
    sheet_name="Year 2009-2010",
    engine="openpyxl"
)

df_2010_2011 = pd.read_excel(
    file_name,
    sheet_name="Year 2010-2011",
    engine="openpyxl"
)

NameError: name 'uploaded' is not defined

In [ ]:
#add a source-year column so we can still identify which sheet each record came from
df_2009_2010 = df_2009_2010.copy()
df_2010_2011 = df_2010_2011.copy()

df_2009_2010["SourceYear"] = "2009-2010"
df_2010_2011["SourceYear"] = "2010-2011"

In [ ]:
#combine them
df_raw = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

In [ ]:
print("Combined shape:", df_raw.shape)
print("Total rows:", len(df_raw))
print("Total columns:", len(df_raw.columns))

Combined shape: (1067371, 9)
Total rows: 1067371
Total columns: 9


In [ ]:
#verify the year counts
print(df_raw["SourceYear"].value_counts())

SourceYear
2010-2011    541910
2009-2010    525461
Name: count, dtype: int64


In [ ]:
#check the date range
print("Start date:", df_raw["InvoiceDate"].min())
print("End date:", df_raw["InvoiceDate"].max())

Start date: 2009-12-01 07:45:00
End date: 2011-12-09 12:50:00


In [ ]:
df_raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010


In [ ]:
df_raw.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,2010-2011
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,2010-2011
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France,2010-2011


**Correct the duplicate removal**

In [ ]:
original_cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

In [ ]:
print(
    "Duplicate rows ignoring SourceYear:",
    df_raw.duplicated(
        subset=original_cols
    ).sum()
)

Duplicate rows ignoring SourceYear: 34335


In [ ]:
df_clean = (
    df_raw
    .drop_duplicates(
        subset=original_cols,
        keep="first"
    )
    .copy()
)

In [ ]:
print("Raw rows:", len(df_raw))
print("Clean rows:", len(df_clean))
print("Total duplicate copies removed:",
      len(df_raw) - len(df_clean))

Raw rows: 1067371
Clean rows: 1033036
Total duplicate copies removed: 34335


In [ ]:
#verify no duplicates left
print(
    "Remaining exact duplicates:",
    df_clean.duplicated(
        subset=original_cols
    ).sum()
)

Remaining exact duplicates: 0


**Rebuild the three datasets from corrected df_clean**

*purchases*

In [ ]:
df_purchase = df_clean[
    (df_clean["Customer ID"].notna()) &
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] >= 0) &
    (~df_clean["Invoice"].astype(str).str.startswith("C"))
].copy()

*Returns/cancellations:*

In [ ]:
df_returns = df_clean[
    (df_clean["Customer ID"].notna()) &
    (df_clean["Quantity"] < 0) &
    (df_clean["Invoice"].astype(str).str.startswith("C"))
].copy()

*Everything else:*

In [ ]:
purchase_idx = df_purchase.index
return_idx = df_returns.index

df_operational = df_clean[
    ~df_clean.index.isin(
        purchase_idx.union(return_idx)
    )
].copy()

In [ ]:
#verify
print("Purchase rows:", len(df_purchase))
print("Return rows:", len(df_returns))
print("Operational/other rows:", len(df_operational))

print(
    "Total:",
    len(df_purchase)
    + len(df_returns)
    + len(df_operational)
)

print("df_clean total:", len(df_clean))

Purchase rows: 779495
Return rows: 18390
Operational/other rows: 235151
Total: 1033036
df_clean total: 1033036


**Rebuild the merchandise purchase dataset**

In [ ]:
special_codes = [
    "POST",
    "DOT",
    "M",
    "AMAZONFEE",
    "BANK CHARGES",
    "B",
    "TEST001",
    "PADS"
]

In [ ]:
#corrected merchandise dataset

df_merchandise = df_purchase[
    ~df_purchase["StockCode"].astype(str).isin(special_codes)
].copy()

In [ ]:
print("Purchase rows before:", len(df_purchase))
print("Merchandise rows after:", len(df_merchandise))
print(
    "Special-code rows removed:",
    len(df_purchase) - len(df_merchandise)
)

Purchase rows before: 779495
Merchandise rows after: 776928
Special-code rows removed: 2567


In [ ]:
print(
    "Special codes remaining:",
    df_merchandise[
        df_merchandise["StockCode"].astype(str).isin(special_codes)
    ].shape[0]
)

Special codes remaining: 0


In [ ]:
#recreate TransactionValue because this is a newly rebuilt dataframe:

df_merchandise["TransactionValue"] = (
    df_merchandise["Quantity"] * df_merchandise["Price"]
)

In [ ]:
print(df_merchandise["TransactionValue"].describe())

count    776928.000000
mean         21.991481
std         225.884868
min           0.000000
25%           4.950000
50%          12.480000
75%          19.800000
max      168469.600000
Name: TransactionValue, dtype: float64


**Create the corrected order-level dataset**

In [ ]:
#group primarily by Invoice:

df_orders = (
    df_merchandise
    .groupby("Invoice", as_index=False)
    .agg(
        CustomerID=("Customer ID", "first"),
        OrderDate=("InvoiceDate", "min"),
        Country=("Country", "first"),
        SourceYear=("SourceYear", "first"),

        OrderValue=("TransactionValue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
        UniqueProducts=("StockCode", "nunique")
    )
)

In [ ]:
OrderDate=("InvoiceDate", "min")

In [ ]:
print("Rows in df_orders:", len(df_orders))
print("Unique invoices:", df_orders["Invoice"].nunique())
print("Unique customers:", df_orders["CustomerID"].nunique())

#Rows in df_orders: 36644 = Unique invoices: 36644

Rows in df_orders: 36644
Unique invoices: 36644
Unique customers: 5863


In [ ]:
#every invoice appears once
print(
    "Duplicate invoices in df_orders:",
    df_orders["Invoice"].duplicated().sum()
)

Duplicate invoices in df_orders: 0


**Inspect the order variables**

In [ ]:
df_orders[
    [
        "OrderValue",
        "TotalQuantity",
        "ProductLines",
        "UniqueProducts"
    ]
].describe()

,OrderValue,TotalQuantity,ProductLines,UniqueProducts
count,36644.000000,36644.000000,36644.000000,36644.000000
mean,466.264523,286.921570,21.202052,20.915320
std,1357.999729,1239.175695,22.965387,22.399956
min,0.000000,1.000000,1.000000,1.000000
25%,158.447500,74.000000,7.000000,7.000000
50%,302.570000,153.000000,15.000000,15.000000
75%,474.022500,288.000000,28.000000,27.000000
max,168469.600000,87167.000000,541.000000,540.000000


In [ ]:
df_orders.head(20)

,Invoice,CustomerID,OrderDate,Country,SourceYear,OrderValue,TotalQuantity,ProductLines,UniqueProducts
0,489434,13085.0,2009-12-01 07:45:00,United Kingdom,2009-2010,505.30,166,8,8
1,489435,13085.0,2009-12-01 07:46:00,United Kingdom,2009-2010,145.80,60,4,4
2,489436,13078.0,2009-12-01 09:06:00,United Kingdom,2009-2010,630.33,193,19,19
3,489437,15362.0,2009-12-01 09:08:00,United Kingdom,2009-2010,310.75,145,23,23
4,489438,18102.0,2009-12-01 09:24:00,United Kingdom,2009-2010,2286.24,826,17,17
5,489439,12682.0,2009-12-01 09:28:00,France,2009-2010,372.30,216,18,18
6,489440,18087.0,2009-12-01 09:43:00,United Kingdom,2009-2010,50.40,16,2,2
7,489441,18087.0,2009-12-01 09:44:00,United Kingdom,2009-2010,344.34,102,4,4
8,489442,13635.0,2009-12-01 09:46:00,United Kingdom,2009-2010,382.37,275,23,23
9,489443,14110.0,2009-12-01 09:50:00,United Kingdom,2009-2010,285.06,120,7,7


In [ ]:
df_orders.nlargest(
    20,
    "OrderValue"
)[
    [
        "Invoice",
        "CustomerID",
        "OrderDate",
        "OrderValue",
        "TotalQuantity",
        "ProductLines",
        "UniqueProducts",
        "Country"
    ]
]

,Invoice,CustomerID,OrderDate,OrderValue,TotalQuantity,ProductLines,UniqueProducts,Country
36611,581483,16446.0,2011-12-09 09:15:00,168469.60,80995,1,1,United Kingdom
20140,541431,12346.0,2011-01-18 10:01:00,77183.60,74215,1,1,United Kingdom
1588,493819,14156.0,2010-01-07 12:34:00,44051.60,25018,94,94,EIRE
26117,556444,15098.0,2011-06-10 15:28:00,38970.00,60,1,1,United Kingdom
13269,524181,17450.0,2010-09-27 16:59:00,33167.80,8820,14,14,United Kingdom
30575,567423,17450.0,2011-09-20 11:05:00,31698.16,12572,12,12,United Kingdom
14452,526934,18102.0,2010-10-14 09:46:00,26007.08,5079,15,15,United Kingdom
9915,515944,18102.0,2010-07-15 15:29:00,22863.36,4992,17,17,United Kingdom
26303,556917,12415.0,2011-06-15 13:37:00,22775.93,15049,138,138,Australia
32594,572209,18102.0,2011-10-21 12:08:00,22206.00,1920,7,7,United Kingdom


**Build the customer-level dataset-one row per customer**

In [ ]:
# For each customer, we will initially create:

# Frequency = number of orders
# Monetary = total amount spent
# AverageOrderValue
# TotalQuantity
# AverageBasketQuantity
# AverageUniqueProducts
# FirstPurchaseDate
# LastPurchaseDate

df_customers = (
    df_orders
    .groupby("CustomerID", as_index=False)
    .agg(
        Frequency=("Invoice", "nunique"),
        Monetary=("OrderValue", "sum"),
        AverageOrderValue=("OrderValue", "mean"),
        TotalQuantity=("TotalQuantity", "sum"),
        AverageBasketQuantity=("TotalQuantity", "mean"),
        AverageUniqueProducts=("UniqueProducts", "mean"),
        FirstPurchaseDate=("OrderDate", "min"),
        LastPurchaseDate=("OrderDate", "max")
    )
)

In [ ]:
print("Rows in customer dataset:", len(df_customers))
print("Unique customers:", df_customers["CustomerID"].nunique())

Rows in customer dataset: 5863
Unique customers: 5863


In [ ]:
df_customers.head(20)

,CustomerID,Frequency,Monetary,AverageOrderValue,TotalQuantity,AverageBasketQuantity,AverageUniqueProducts,FirstPurchaseDate,LastPurchaseDate
0,12346.0,4,77353.96,19338.490000,74240,18560.000000,6.500000,2009-12-18 10:55:00,2011-01-18 10:01:00
1,12347.0,8,4921.53,615.191250,2967,370.875000,27.750000,2010-10-31 14:20:00,2011-12-07 15:52:00
2,12348.0,5,1658.40,331.680000,2704,540.800000,8.400000,2010-09-27 14:59:00,2011-09-25 13:13:00
3,12349.0,3,3678.69,1226.230000,1621,540.333333,57.333333,2010-04-29 13:20:00,2011-11-21 09:51:00
4,12350.0,1,294.40,294.400000,196,196.000000,16.000000,2011-02-02 16:01:00,2011-02-02 16:01:00
5,12351.0,1,300.93,300.930000,261,261.000000,21.000000,2010-11-29 15:23:00,2010-11-29 15:23:00
6,12352.0,9,1729.54,192.171111,714,79.333333,10.555556,2010-11-12 10:20:00,2011-11-03 14:37:00
7,12353.0,2,406.76,203.380000,212,106.000000,12.000000,2010-10-27 12:44:00,2011-05-19 17:47:00
8,12354.0,1,1079.40,1079.400000,530,530.000000,58.000000,2011-04-21 13:11:00,2011-04-21 13:11:00
9,12355.0,2,947.61,473.805000,543,271.500000,17.500000,2010-05-21 11:59:00,2011-05-09 13:49:00


In [ ]:
df_customers[
    [
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "TotalQuantity",
        "AverageBasketQuantity",
        "AverageUniqueProducts"
    ]
].describe()

,Frequency,Monetary,AverageOrderValue,TotalQuantity,AverageBasketQuantity,AverageUniqueProducts
count,5863.000000,5863.00000,5863.000000,5863.000000,5863.000000,5863.000000
mean,6.250043,2914.17315,382.769145,1793.272045,255.162760,21.226522
std,12.790250,14327.73229,1233.906830,8891.501690,1456.701726,18.275722
min,1.000000,0.00000,0.000000,1.000000,1.000000,1.000000
25%,1.000000,338.55500,175.745357,188.000000,91.763889,9.710084
50%,3.000000,854.99000,277.516000,484.000000,154.724138,17.000000
75%,7.000000,2236.94500,410.516250,1351.000000,259.090909,27.000000
max,378.000000,580987.04000,84236.250000,367712.000000,87167.000000,297.882353


**Add Recency and Customer Tenure**

In [ ]:
#reference date just after the last transaction date:
#Recency = how many days since the customer's most recent purchase.
#Smaller recency usually means the customer purchased more recently.
#CustomerTenure = how long the customer's observed purchase history spans.

reference_date = df_orders["OrderDate"].max() + pd.Timedelta(days=1)

print("Reference date:", reference_date)

Reference date: 2011-12-10 12:50:00


In [ ]:
df_customers["Recency"] = (
    reference_date - df_customers["LastPurchaseDate"]
).dt.days

df_customers["CustomerTenure"] = (
    df_customers["LastPurchaseDate"]
    - df_customers["FirstPurchaseDate"]
).dt.days

In [ ]:
df_customers[
    [
        "CustomerID",
        "FirstPurchaseDate",
        "LastPurchaseDate",
        "Recency",
        "CustomerTenure"
    ]
].head(20)

,CustomerID,FirstPurchaseDate,LastPurchaseDate,Recency,CustomerTenure
0,12346.0,2009-12-18 10:55:00,2011-01-18 10:01:00,326,395
1,12347.0,2010-10-31 14:20:00,2011-12-07 15:52:00,2,402
2,12348.0,2010-09-27 14:59:00,2011-09-25 13:13:00,75,362
3,12349.0,2010-04-29 13:20:00,2011-11-21 09:51:00,19,570
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00,310,0
5,12351.0,2010-11-29 15:23:00,2010-11-29 15:23:00,375,0
6,12352.0,2010-11-12 10:20:00,2011-11-03 14:37:00,36,356
7,12353.0,2010-10-27 12:44:00,2011-05-19 17:47:00,204,204
8,12354.0,2011-04-21 13:11:00,2011-04-21 13:11:00,232,0
9,12355.0,2010-05-21 11:59:00,2011-05-09 13:49:00,214,353


In [ ]:
df_customers[
    ["Recency", "CustomerTenure"]
].describe()

,Recency,CustomerTenure
count,5863.000000,5863.000000
mean,200.956848,273.107965
std,209.216692,258.522024
min,1.000000,0.000000
25%,26.000000,0.000000
50%,96.000000,221.000000
75%,379.000000,511.000000
max,739.000000,738.000000


**Add Product Diversity per Customer**-measure how many different products each customer purchased across the whole dataset.
Customer A repeatedly buys only 2 products
Customer B buys 50 different products

In [ ]:
customer_product_diversity = (
    df_merchandise
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "Customer ID": "CustomerID",
            "StockCode": "ProductDiversity"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_product_diversity,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "Monetary",
        "ProductDiversity"
    ]
].head(20)

,CustomerID,Frequency,Monetary,ProductDiversity
0,12346.0,4,77353.96,26
1,12347.0,8,4921.53,126
2,12348.0,5,1658.40,24
3,12349.0,3,3678.69,137
4,12350.0,1,294.40,16
5,12351.0,1,300.93,21
6,12352.0,9,1729.54,68
7,12353.0,2,406.76,23
8,12354.0,1,1079.40,58
9,12355.0,2,947.61,35


In [ ]:
df_customers["ProductDiversity"].describe()

,ProductDiversity
count,5863.000000
mean,82.049974
std,116.464707
min,1.000000
25%,19.000000
50%,45.000000
75%,103.000000
max,2548.000000


**Prepare Customer Return Behaviour**

In [ ]:
#remove the same non-merchandise special codes from the return dataset:

df_returns_merchandise = df_returns[
    ~df_returns["StockCode"].astype(str).isin(special_codes)
].copy()

In [ ]:
print("All return rows:", len(df_returns))
print("Merchandise return rows:", len(df_returns_merchandise))
print(
    "Special return rows removed:",
    len(df_returns) - len(df_returns_merchandise)
)

All return rows: 18390
Merchandise return rows: 17802
Special return rows removed: 588


In [ ]:
#return quantities are negative, create a positive quantity representing the number of items returned:

df_returns_merchandise["ReturnedQuantity"] = (
    df_returns_merchandise["Quantity"].abs()
)

In [ ]:
#create the monetary value of the return:

df_returns_merchandise["ReturnValue"] = (
    df_returns_merchandise["ReturnedQuantity"]
    * df_returns_merchandise["Price"]
)

In [ ]:
df_returns_merchandise[
    [
        "Invoice",
        "Customer ID",
        "StockCode",
        "Quantity",
        "ReturnedQuantity",
        "Price",
        "ReturnValue"
    ]
].head(20)

,Invoice,Customer ID,StockCode,Quantity,ReturnedQuantity,Price,ReturnValue
178,C489449,16321.0,22087,-12,12,2.95,35.40
179,C489449,16321.0,85206A,-6,6,1.65,9.90
180,C489449,16321.0,21895,-4,4,4.25,17.00
181,C489449,16321.0,21896,-6,6,2.10,12.60
182,C489449,16321.0,22083,-12,12,2.95,35.40
183,C489449,16321.0,21871,-12,12,1.25,15.00
184,C489449,16321.0,84946,-12,12,1.25,15.00
185,C489449,16321.0,84970S,-24,24,0.85,20.40
186,C489449,16321.0,22090,-12,12,2.95,35.40
196,C489459,17592.0,90200A,-3,3,4.25,12.75


In [ ]:
print(
    df_returns_merchandise[
        ["ReturnedQuantity", "ReturnValue"]
    ].describe()
)

       ReturnedQuantity    ReturnValue
count      17802.000000   17802.000000
mean          26.255702      41.166469
std          839.413685    1394.330344
min            1.000000       0.120000
25%            1.000000       3.750000
50%            2.000000       8.500000
75%            6.000000      17.000000
max        80995.000000  168469.600000


**Create Customer-Level Return Features**

In [ ]:
customer_returns = (
    df_returns_merchandise
    .groupby("Customer ID", as_index=False)
    .agg(
        ReturnTransactions=("Invoice", "nunique"),
        ReturnedQuantity=("ReturnedQuantity", "sum"),
        TotalReturnValue=("ReturnValue", "sum")
    )
    .rename(
        columns={
            "Customer ID": "CustomerID"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_returns,
    on="CustomerID",
    how="left"
)

In [ ]:
#Customers who never returned anything will now have missing values, so replace those with zero:

return_columns = [
    "ReturnTransactions",
    "ReturnedQuantity",
    "TotalReturnValue"
]

df_customers[return_columns] = (
    df_customers[return_columns].fillna(0)
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "Monetary",
        "ReturnTransactions",
        "ReturnedQuantity",
        "TotalReturnValue"
    ]
].head(20)

,CustomerID,Frequency,Monetary,ReturnTransactions,ReturnedQuantity,TotalReturnValue
0,12346.0,4,77353.96,3.0,74217.0,77288.10
1,12347.0,8,4921.53,0.0,0.0,0.00
2,12348.0,5,1658.40,0.0,0.0,0.00
3,12349.0,3,3678.69,1.0,5.0,24.15
4,12350.0,1,294.40,0.0,0.0,0.00
5,12351.0,1,300.93,0.0,0.0,0.00
6,12352.0,9,1729.54,1.0,63.0,120.33
7,12353.0,2,406.76,0.0,0.0,0.00
8,12354.0,1,1079.40,0.0,0.0,0.00
9,12355.0,2,947.61,0.0,0.0,0.00


In [ ]:
df_customers[
    [
        "ReturnTransactions",
        "ReturnedQuantity",
        "TotalReturnValue"
    ]
].describe()

,ReturnTransactions,ReturnedQuantity,TotalReturnValue
count,5863.000000,5863.000000,5863.000000
mean,1.267952,79.622207,124.714687
std,3.484529,1871.599394,2505.802488
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,1.000000,8.000000,27.905000
max,95.000000,87919.000000,168469.600000


In [ ]:
#check how many customers made at least one return:
#Monetary = gross merchandise purchases
#TotalReturnValue = merchandise cancellations/returns

customers_with_returns = (
    df_customers["ReturnTransactions"] > 0
).sum()

print("Customers with at least one return:", customers_with_returns)

print(
    "Customers with no returns:",
    (df_customers["ReturnTransactions"] == 0).sum()
)

Customers with at least one return: 2440
Customers with no returns: 3423


**Create Return Rate**

In [ ]:
df_customers["ReturnRate"] = (
    df_customers["TotalReturnValue"] /
    df_customers["Monetary"]
)

In [ ]:
df_customers["ReturnRate"] = (
    df_customers["ReturnRate"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [ ]:
df_customers["ReturnRate"].describe()

,ReturnRate
count,5863.000000
mean,0.024302
std,0.096531
min,0.000000
25%,0.000000
50%,0.000000
75%,0.015104
max,2.858386


In [ ]:
df_customers.nlargest(
    20,
    "ReturnRate"
)[
    [
        "CustomerID",
        "Monetary",
        "TotalReturnValue",
        "ReturnTransactions",
        "ReturnRate"
    ]
]

,CustomerID,Monetary,TotalReturnValue,ReturnTransactions,ReturnRate
3541,15935.0,108.04,308.82,1.0,2.858386
739,13091.0,1108.80,2452.04,3.0,2.211436
759,13112.0,20.60,26.04,2.0,1.264078
120,12467.0,132.80,132.80,1.0,1.000000
211,12558.0,269.96,269.96,1.0,1.000000
260,12607.0,1579.51,1579.51,1.0,1.000000
419,12768.0,168.69,168.69,1.0,1.000000
424,12773.0,124.25,124.25,1.0,1.000000
810,13163.0,17.65,17.65,1.0,1.000000
1006,13364.0,134.96,134.96,1.0,1.000000


In [ ]:
df_customers.nlargest(
    20,
    "ReturnRate"
)[
    [
        "CustomerID",
        "Monetary",
        "TotalReturnValue",
        "ReturnTransactions",
        "ReturnRate"
    ]
]

,CustomerID,Monetary,TotalReturnValue,ReturnTransactions,ReturnRate
3541,15935.0,108.04,308.82,1.0,2.858386
739,13091.0,1108.80,2452.04,3.0,2.211436
759,13112.0,20.60,26.04,2.0,1.264078
120,12467.0,132.80,132.80,1.0,1.000000
211,12558.0,269.96,269.96,1.0,1.000000
260,12607.0,1579.51,1579.51,1.0,1.000000
419,12768.0,168.69,168.69,1.0,1.000000
424,12773.0,124.25,124.25,1.0,1.000000
810,13163.0,17.65,17.65,1.0,1.000000
1006,13364.0,134.96,134.96,1.0,1.000000


**Create Purchase Interval Features**

Measure how regularly customers purchase.

For customers with multiple orders, we can calculate the time gaps between purchases.

In [ ]:
#sort the orders:

df_orders = df_orders.sort_values(
    ["CustomerID", "OrderDate"]
).copy()

In [ ]:
#calculate the gap between consecutive orders:

df_orders["DaysSincePreviousOrder"] = (
    df_orders
    .groupby("CustomerID")["OrderDate"]
    .diff()
    .dt.days
)

In [ ]:
customer_intervals = (
    df_orders
    .groupby("CustomerID")["DaysSincePreviousOrder"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "DaysSincePreviousOrder": "AveragePurchaseInterval"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_intervals,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "AveragePurchaseInterval"
    ]
].head(20)

,CustomerID,Frequency,AveragePurchaseInterval
0,12346.0,4,131.666667
1,12347.0,8,57.000000
2,12348.0,5,90.500000
3,12349.0,3,285.000000
4,12350.0,1,NaN
5,12351.0,1,NaN
6,12352.0,9,44.125000
7,12353.0,2,204.000000
8,12354.0,1,NaN
9,12355.0,2,353.000000


In [ ]:
df_customers["AveragePurchaseInterval"].describe()

,AveragePurchaseInterval
count,4236.000000
mean,103.057291
std,99.337160
min,0.000000
25%,38.891667
50%,72.400000
75%,130.500000
max,714.000000


**Create Active Purchase Days**

In [ ]:
df_orders["PurchaseDate"] = (
    df_orders["OrderDate"].dt.date
)

In [ ]:
#calculate unique purchase days for every customer:

active_days = (
    df_orders
    .groupby("CustomerID")["PurchaseDate"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "PurchaseDate": "ActivePurchaseDays"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    active_days,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers["OrdersPerActiveDay"] = (
    df_customers["Frequency"] /
    df_customers["ActivePurchaseDays"]
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "ActivePurchaseDays",
        "OrdersPerActiveDay"
    ]
].head(20)

,CustomerID,Frequency,ActivePurchaseDays,OrdersPerActiveDay
0,12346.0,4,4,1.0
1,12347.0,8,8,1.0
2,12348.0,5,5,1.0
3,12349.0,3,3,1.0
4,12350.0,1,1,1.0
5,12351.0,1,1,1.0
6,12352.0,9,9,1.0
7,12353.0,2,2,1.0
8,12354.0,1,1,1.0
9,12355.0,2,2,1.0


In [ ]:
df_customers[
    [
        "ActivePurchaseDays",
        "OrdersPerActiveDay"
    ]
].describe()

,ActivePurchaseDays,OrdersPerActiveDay
count,5863.000000,5863.000000
mean,5.614020,1.065626
std,9.933415,0.206353
min,1.000000,1.000000
25%,1.000000,1.000000
50%,3.000000,1.000000
75%,6.000000,1.000000
max,250.000000,6.739130


**Final quality check of the customer-level dataset**

In [ ]:
print("Rows:", df_customers.shape[0])
print("Columns:", df_customers.shape[1])
print("Unique customers:", df_customers["CustomerID"].nunique())

Rows: 5863
Columns: 19
Unique customers: 5863


In [ ]:
print(
    "Duplicate Customer IDs:",
    df_customers["CustomerID"].duplicated().sum()
)

Duplicate Customer IDs: 0


In [ ]:
df_customers.isnull().sum().sort_values(ascending=False)

,0
AveragePurchaseInterval,1627
Frequency,0
CustomerID,0
Monetary,0
AverageOrderValue,0
AverageBasketQuantity,0
TotalQuantity,0
FirstPurchaseDate,0
LastPurchaseDate,0
Recency,0


In [ ]:
#Check infinite values
numeric_cols = df_customers.select_dtypes(include=np.number).columns

print(
    "Total infinite values:",
    np.isinf(df_customers[numeric_cols]).sum().sum()
)

Total infinite values: 0


In [ ]:
print(df_customers.columns.tolist())

['CustomerID', 'Frequency', 'Monetary', 'AverageOrderValue', 'TotalQuantity', 'AverageBasketQuantity', 'AverageUniqueProducts', 'FirstPurchaseDate', 'LastPurchaseDate', 'Recency', 'CustomerTenure', 'ProductDiversity', 'ReturnTransactions', 'ReturnedQuantity', 'TotalReturnValue', 'ReturnRate', 'AveragePurchaseInterval', 'ActivePurchaseDays', 'OrdersPerActiveDay']


In [ ]:
## Log Transformation of Skewed Customer Features

# Customer behavioural variables are highly right-skewed due to a small number
# of customers with very high purchasing activity.

# Log transformation is applied to suitable non-negative numerical variables
# before parametric statistical inference and modelling.

# Original variables are retained for interpretation.

In [ ]:
#Check skewness before transforming



skew_check_cols = [
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "AverageBasketQuantity",
    "AverageUniqueProducts",
    "Recency",
    "CustomerTenure",
    "ProductDiversity",
    "ReturnTransactions",
    "ReturnedQuantity",
    "TotalReturnValue",
    "ReturnRate",
    "AveragePurchaseInterval",
    "ActivePurchaseDays",
    "OrdersPerActiveDay"
]

skew_before = (
    df_customers[skew_check_cols]
    .skew()
    .sort_values(ascending=False)
)

print("Skewness before transformation:")
print(skew_before)

Skewness before transformation:
TotalReturnValue           57.108844
AverageOrderValue          55.157848
AverageBasketQuantity      45.357365
ReturnedQuantity           42.203468
Monetary                   25.309836
TotalQuantity              21.853676
Frequency                  12.143308
ReturnRate                 11.785514
ReturnTransactions         11.199608
ActivePurchaseDays          9.873191
OrdersPerActiveDay          8.488859
ProductDiversity            6.226626
AverageUniqueProducts       3.278915
AveragePurchaseInterval     2.172379
Recency                     0.891870
CustomerTenure              0.387853
dtype: float64


In [ ]:
# Features selected for log transformation
log_transform_cols = [
    "TotalReturnValue",
    "AverageOrderValue",
    "AverageBasketQuantity",
    "ReturnedQuantity",
    "Monetary",
    "TotalQuantity",
    "Frequency",
    "ReturnRate",
    "ReturnTransactions",
    "ActivePurchaseDays",
    "OrdersPerActiveDay",
    "ProductDiversity",
    "AverageUniqueProducts"
]

# Create new log-transformed columns
# Keep the original columns unchanged
for col in log_transform_cols:
    df_customers["Log_" + col] = np.log1p(df_customers[col])

In [ ]:
log_cols = ["Log_" + col for col in log_transform_cols]

skew_after = (
    df_customers[log_cols]
    .skew()
    .sort_values(ascending=False)
)

print("Skewness after log transformation:")
print(skew_after)

Skewness after log transformation:
Log_ReturnRate               7.688101
Log_OrdersPerActiveDay       4.711539
Log_ReturnedQuantity         1.529507
Log_ReturnTransactions       1.480841
Log_Frequency                1.004043
Log_ActivePurchaseDays       0.972718
Log_TotalReturnValue         0.964927
Log_Monetary                 0.243648
Log_TotalQuantity           -0.009751
Log_AverageOrderValue       -0.028619
Log_ProductDiversity        -0.268636
Log_AverageBasketQuantity   -0.377531
Log_AverageUniqueProducts   -0.451830
dtype: float64


In [ ]:
# AverageOrderValue       ← original, business interpretation
# Log_AverageOrderValue   ← statistical analysis when needed



log_cols = ["Log_" + col for col in log_transform_cols]

print("Infinite values:",
      np.isinf(df_customers[log_cols]).sum().sum())

print("\nMissing values:")
print(df_customers[log_cols].isnull().sum())

Infinite values: 0

Missing values:
Log_TotalReturnValue         0
Log_AverageOrderValue        0
Log_AverageBasketQuantity    0
Log_ReturnedQuantity         0
Log_Monetary                 0
Log_TotalQuantity            0
Log_Frequency                0
Log_ReturnRate               0
Log_ReturnTransactions       0
Log_ActivePurchaseDays       0
Log_OrdersPerActiveDay       0
Log_ProductDiversity         0
Log_AverageUniqueProducts    0
dtype: int64


In [ ]:
df_clean.to_csv(
    "/content/online_retail_clean.csv",
    index=False
)

df_merchandise.to_csv(
    "/content/online_retail_merchandise.csv",
    index=False
)

df_orders.to_csv(
    "/content/order_level_data.csv",
    index=False
)

df_customers.to_csv(
    "/content/customer_level_data.csv",
    index=False
)

In [ ]:
print("Saved successfully!")

print("Clean transactions:", df_clean.shape)
print("Merchandise transactions:", df_merchandise.shape)
print("Orders:", df_orders.shape)
print("Customers:", df_customers.shape)

Saved successfully!
Clean transactions: (1033036, 9)
Merchandise transactions: (776928, 10)
Orders: (36644, 11)
Customers: (5863, 32)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive"

'2025-07-21_09-35-34 - user.mp4'
'At the beginning of this note add a mind map to g... (1).gdoc'
'At the beginning of this note add a mind map to g... (2).gdoc'
'At the beginning of this note add a mind map to g....gdoc'
'Chanuthi_Jayasingha (1).pdf'
'Chanuthi Jayasingha.pdf'
 Chanuthi_Jayasingha.pdf
 Cognitive_IVMS_Final_Report.gdoc
'Colab Notebooks'
'DWBI Self-Study Tutorial Plan.gdoc'
'Google AI Studio'
'Here are 3 unique, localized data science projects designed to grab the attention of Sri Lankan recruiters in both tech and non-tech sectors.gdoc'
'Inventory Hub Use Case Scenario Generation.gsheet'
'Inventory usecase.drawio'
'Inventory usecase-Page-1.drawio.png'
'Lec 1-PDP.gdoc'
'SKU Registration Test Cases Generation.gsheet'
'Start research.gdoc'
'Statistical Modelling Project'
'Statistical Modelling Projectcustomer_level_data'
'Statistical Modelling Projectonline_retail_clean'
'Statistical Modelling Projectonline_retail_merchandise'
'Statistical Modelling Projectorder_level_data'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_path = "/content/drive/MyDrive/Statistical Modelling Project/02_Clean_Data/"

In [ ]:
df_clean.to_csv(
    save_path + "online_retail_clean",
    index=False
)

df_merchandise.to_csv(
    save_path + "online_retail_merchandise",
    index=False
)

df_orders.to_csv(
    save_path + "order_level_data",
    index=False
)

df_customers.to_csv(
    save_path + "customer_level_data",
    index=False
)

print("Saved successfully to:")
print(save_path)

Saved successfully to:
/content/drive/MyDrive/Statistical Modelling Project


In [ ]:
!ls "/content/drive/MyDrive/Statistical Modelling Project/"

 00_Project_Documents
 02_Clean_Data
 03_Notebooks
 04_Final_Model_Outputs
'Elastic Net Regression'
'Final_Three_Model_Outputs- logisctic decisiontree XGboost'
'Lasso Logistic Regression'
'Model training - test train spliit'
 Old_Task_Work
'Ridge logistic Regression'
 Untitled0.ipynb
